# RSNA Knee — build the full training cache (CPU)

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-001** | `hlly34/rsna-knee-cache` | full training cache: 4,407 studies, 256 px, 72 slices, 140 mm, canonical knee |

Decoding ~570 GB of DICOM is CPU work, so it runs here without spending GPU quota. The output (`cache_256/<StudyInstanceUID>.npz`, compressed to fit Kaggle's 20 GB output limit) is attached to the training notebooks.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-001", "notebook": "hlly34/rsna-knee-cache", "what": "full training cache: 4,407 studies, 256 px, 72 slices, 140 mm, canonical knee"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


## What happens to each study
1. **Pick series** — 6 slots: sagittal/coronal/axial × fluid-sensitive/not (same-plane fallback).
2. **Order slices** by `ImagePositionPatient` along the slice normal (file names are random).
3. **Canonical knee** — side from the image-centre x coordinate (tag as fallback); left knees are mirrored, sagittal runs lateral → medial.
4. **Intensity** — per-series 1–99th percentile → uint8.
5. **Geometry** — 140 mm physical centre crop → 256×256; 72 slices sampled over 4–96% of each series (20/10/16/8/14/4 per slot).
6. **Save** `image (72,256,256)`, `slot`, `valid`, `side`.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
DATA = "/kaggle/tmp/data"; os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/raw"):
    os.symlink(COMP, f"{DATA}/raw")
ENV = dict(os.environ, DATA_DIR=DATA, PYTHONPATH=CODE)
t = time.time()
subprocess.run([sys.executable, f"{CODE}/scripts/preprocess.py", "exp=effnet_224", "preprocess.compress=true",
                f"preprocess.num_workers={os.cpu_count()}", "paths.cache_dir=/kaggle/working/cache_256"],
               env=ENV, check=True)
print(f"preprocess {(time.time() - t) / 60:.1f} min")

## Sanity checks

In [ ]:
import numpy as np
files = sorted(glob.glob("/kaggle/working/cache_256/*.npz"))
sides, cover = [], []
for f in files[::10]:
    z = np.load(f); sides.append(int(z["side"])); cover.append(z["valid"].mean())
size = sum(os.path.getsize(f) for f in files) / 2**30
EXP.update(studies=len(files), size_gib=round(size, 2), side_R=sides.count(1), side_L=sides.count(-1),
           side_unknown=sides.count(0), slot_coverage=round(float(np.mean(cover)), 4))
print(EXP)
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)